# Train ECHO_model Step by Step
This notebook uses the standalone package only.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
PACKAGE_ROOT = ROOT if (ROOT / 'echo_model').exists() else ROOT.parent
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

from echo_model.data import build_demo_dataset
from echo_model.losses import RmseLossComb
from echo_model.rnn import MultiInv_DynamicSimHydModelSix_Physical_ClosedSnowaSrzSIMHYDSimple
from echo_model.train import train_model
from echo_model.evaluate import evaluate_model
from echo_model.train_utils import get_device


In [ ]:
dataset = build_demo_dataset(PACKAGE_ROOT / 'demo_data', bufftime=30)
print('x_train', dataset['x_train'].shape)
print('z_train', dataset['z_train'].shape)
print('attrs', dataset['attrs'].shape)
print('x_eval', dataset['x_eval'].shape)
print('obs_test', dataset['obs_test'].shape)

In [ ]:
model = MultiInv_DynamicSimHydModelSix_Physical_ClosedSnowaSrzSIMHYDSimple(
    ninv=dataset['z_train'].shape[-1] + dataset['attrs'].shape[-1],
    nmul=2,
    nattr=dataset['attrs'].shape[-1],
    hiddeninv=16,
    inittime=30,
    routOpt=True,
    comprout=False,
    compwts=True,
    lgdyn=True,
    lgdynweight=0.6,
    dynamic_sq=True,
    dynamic_etgam=True,
    dynamic_partition=True,
    dynamic_cfmax_snow=True,
    dynamic_routing_scale=False,
    component_routing=True,
)
print(model.__class__.__name__)
print('staticOut weight shape', tuple(model.staticOut.weight.shape))

In [ ]:
history = train_model(
    model,
    dataset,
    RmseLossComb(alpha=0.25),
    epochs=1,
    batch_size=2,
    rho=60,
    bufftime=30,
    max_iter_ep=2,
    out_dir=PACKAGE_ROOT / 'outputs' / 'notebook_smoke',
    seed=123,
    use_gpu=False,
)
history

In [ ]:
device = get_device(use_gpu=False)
rows, pred = evaluate_model(model, dataset, device)
rows[:3]

In [ ]:
sorted((ROOT / 'outputs' / 'notebook_smoke').glob('*'))[:10]